In [ ]:
cd ~/Documents/GrowingNetwork/experimental_grow

In [ ]:
import torch
from pathlib import Path
from gromo.containers.resnet import ResNetBasicBlock

In [ ]:
root_location = Path(
    "/run/user/674139/gvfs/sftp:host=titanic/home/tau/trudkiew/tau_frugal/trudkiew/mlruns"
)
post_root_location = "artifacts/best_model/data/model.pth"

experiment_id = "216811188622439851"

fogro_run_ids = [
    "d9a65f330b82422782a2772b057aca21",
    "207aba1f13cb44e8940ac9e1f8cf3af4",
    "b317fea820774b66ab3ff77b0d1ba54f",
    "80bf418e34a84869889425e7b456850e",
    "2cb53647d7ba4d54900cff1719930245",
]

random_run_ids = [
    "caf0c468775b4183a94c5e062793d795",
    "86debc2e08db49149bb1add11a1b512a",
    "da975ec270e142ef8e851cc0d87b5467",
    "117a61270ecf445d9e240eda3ce6e4c1",
    "ccab143ec576444295fc0a09cd4b4c9c",
]

full_run_ids = [
    "11ae2d3670d643518515eca74e9e94e6",
    "3c0b687a43674a77b8ea1c426757b1f6",
    "b187736df8474a4485034f78134a07ed",
    "9bad761894854a718fd2568360617e7d",
    "12b7e63ac6274298bfe27abda7d53f68",
]


We have 3 different models saved: fogro_model, random_model, and full_model. Let's load them and inspect the batch normalization layers to see how their running means and variances differ.

In [ ]:
run_idx = 0

fogro_model = torch.load(
    root_location / experiment_id / fogro_run_ids[run_idx] / post_root_location
)
random_model = torch.load(
    root_location / experiment_id / random_run_ids[run_idx] / post_root_location
)
full_model = torch.load(
    root_location / experiment_id / full_run_ids[run_idx] / post_root_location
)
fogro_model

In [ ]:
hidden_channels = []
for stage in fogro_model.stages:
    hidden_channels.append(tuple(layer.hidden_neurons for layer in stage))
print(hidden_channels)

We are intrested in batch norm layers in the middle of each block.
```python
model.stages[i][j].first_layer.post_layer_function[0]
```
accesses the batch norm layer in the stage `i` and block `j` (4 stages with 2 blocks each).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def get_batchnorm_stats(model):
    """Extract all BatchNorm attributes from the model."""
    stats = {}
    for stage_idx, stage in enumerate(model.stages):
        for block_idx, block in enumerate(stage):
            bn_layer = block.first_layer.post_layer_function[0]
            key = f"Stage {stage_idx}, Block {block_idx}"
            stats[key] = {
                "running_mean": bn_layer.running_mean.cpu().numpy(),
                "running_var": bn_layer.running_var.cpu().numpy(),
                "weight": bn_layer.weight.detach().cpu().numpy(),
                "bias": bn_layer.bias.detach().cpu().numpy(),
                "num_channels": bn_layer.num_features,
            }
    return stats


fogro_stats = get_batchnorm_stats(fogro_model)
random_stats = get_batchnorm_stats(random_model)
full_stats = get_batchnorm_stats(full_model)

print("Extracted BatchNorm stats from all models")
print(f"Number of layers analyzed: {len(fogro_stats)}")
print(f"Attributes extracted: running_mean, running_var, weight, bias")

In [ ]:
%matplotlib widget
# Plot running_var evolution across channels for all models
# Note: Models may have different numbers of channels per layer

fig, axes = plt.subplots(4, 2, figsize=(14, 16))
axes = axes.flatten()

layer_names = list(fogro_stats.keys())

for idx, layer_name in enumerate(layer_names):
    ax = axes[idx]

    fogro_var = fogro_stats[layer_name]["running_var"]
    random_var = random_stats[layer_name]["running_var"]
    full_var = full_stats[layer_name]["running_var"]

    # Each model may have different number of channels
    fogro_channels = np.arange(len(fogro_var))
    random_channels = np.arange(len(random_var))
    full_channels = np.arange(len(full_var))

    ax.plot(
        fogro_channels,
        fogro_var,
        "b-",
        label=f"FoGro ({len(fogro_var)} ch)",
        alpha=0.8,
        linewidth=1.5,
    )
    ax.plot(
        random_channels,
        random_var,
        "r-",
        label=f"Random ({len(random_var)} ch)",
        alpha=0.8,
        linewidth=1.5,
    )
    ax.plot(
        full_channels,
        full_var,
        "g-",
        label=f"Full ({len(full_var)} ch)",
        alpha=0.8,
        linewidth=1.5,
    )

    ax.set_xlabel("Channel Index")
    ax.set_ylabel("Running Variance")
    ax.set_title(f"{layer_name}")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle("Running Variance Evolution Across Channels", y=1.02, fontsize=14)
plt.show()

## Analysis: Channel Order Matters for FoGro and Random Models

In growing networks (FoGro) and randomly initialized models, channels are added incrementally:
- **Earlier channels** (lower indices) were present from the beginning of training
- **Later channels** (higher indices) were added more recently

This should create a pattern in `running_var` where earlier channels have more "settled" statistics.

In [ ]:
# Detailed comparison: Show variance trend from first to last channels
# Normalize channel indices to [0, 1] for comparison across models with different channel counts

fig, axes = plt.subplots(4, 2, figsize=(14, 16))
axes = axes.flatten()

for idx, layer_name in enumerate(fogro_stats.keys()):
    ax = axes[idx]

    fogro_var = fogro_stats[layer_name]["running_var"]
    random_var = random_stats[layer_name]["running_var"]
    full_var = full_stats[layer_name]["running_var"]

    # Normalize channel indices to [0, 1] for comparison
    fogro_norm_idx = np.linspace(0, 1, len(fogro_var))
    random_norm_idx = np.linspace(0, 1, len(random_var))
    full_norm_idx = np.linspace(0, 1, len(full_var))

    # Compute rolling average to see trends
    def smooth(data, window_frac=0.1):
        window = max(1, int(len(data) * window_frac))
        return np.convolve(data, np.ones(window) / window, mode="valid")

    fogro_smooth = smooth(fogro_var)
    random_smooth = smooth(random_var)
    full_smooth = smooth(full_var)

    fogro_smooth_idx = np.linspace(0, 1, len(fogro_smooth))
    random_smooth_idx = np.linspace(0, 1, len(random_smooth))
    full_smooth_idx = np.linspace(0, 1, len(full_smooth))

    ax.plot(fogro_smooth_idx, fogro_smooth, "b-", label="FoGro", linewidth=2)
    ax.plot(random_smooth_idx, random_smooth, "r-", label="Random", linewidth=2)
    ax.plot(full_smooth_idx, full_smooth, "g-", label="Full", linewidth=2)

    ax.set_xlabel("Normalized Channel Index (0=first, 1=last)")
    ax.set_ylabel("Running Variance (smoothed)")
    ax.set_title(f"{layer_name}")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle(
    "Smoothed Running Variance Trend: First → Last Channels", y=1.02, fontsize=14
)
plt.show()

In [ ]:
# Statistical summary: Compare first half vs second half of channels
print("=" * 80)
print("Statistical Summary: First Half vs Second Half of Channels")
print("=" * 80)

for layer_name in fogro_stats.keys():
    fogro_var = fogro_stats[layer_name]["running_var"]
    random_var = random_stats[layer_name]["running_var"]
    full_var = full_stats[layer_name]["running_var"]

    n = len(fogro_var)
    mid = n // 2

    print(f"\n{layer_name} ({n} channels):")
    print("-" * 60)

    for name, var in [("FoGro", fogro_var), ("Random", random_var), ("Full", full_var)]:
        first_half_mean = var[:mid].mean()
        second_half_mean = var[mid:].mean()
        first_half_std = var[:mid].std()
        second_half_std = var[mid:].std()
        ratio = (
            second_half_mean / first_half_mean if first_half_mean > 0 else float("inf")
        )

        print(
            f"  {name:8s}: First half μ={first_half_mean:.4f}±{first_half_std:.4f}, "
            f"Second half μ={second_half_mean:.4f}±{second_half_std:.4f}, "
            f"Ratio={ratio:.3f}"
        )

In [ ]:
# Correlation analysis: Is there a monotonic trend in running_var with channel index?
from scipy import stats

print("=" * 80)
print("Correlation Analysis: Running Variance vs Channel Index")
print("(Spearman correlation - negative = variance DECREASES with channel index)")
print("=" * 80)

correlations = {"FoGro": [], "Random": [], "Full": []}

for layer_name in fogro_stats.keys():
    fogro_var = fogro_stats[layer_name]["running_var"]
    random_var = random_stats[layer_name]["running_var"]
    full_var = full_stats[layer_name]["running_var"]

    # Use individual channel indices for each model
    fogro_channels = np.arange(len(fogro_var))
    random_channels = np.arange(len(random_var))
    full_channels = np.arange(len(full_var))

    fogro_corr, fogro_p = stats.spearmanr(fogro_channels, fogro_var)
    random_corr, random_p = stats.spearmanr(random_channels, random_var)
    full_corr, full_p = stats.spearmanr(full_channels, full_var)

    correlations["FoGro"].append(fogro_corr)
    correlations["Random"].append(random_corr)
    correlations["Full"].append(full_corr)

    print(f"\n{layer_name}:")
    print(f"  FoGro:  ρ={fogro_corr:+.3f} (p={fogro_p:.2e})")
    print(f"  Random: ρ={random_corr:+.3f} (p={random_p:.2e})")
    print(f"  Full:   ρ={full_corr:+.3f} (p={full_p:.2e})")

In [ ]:
# Summary bar plot of correlations
fig, ax = plt.subplots(figsize=(12, 6))

layer_names = list(fogro_stats.keys())
x = np.arange(len(layer_names))
width = 0.25

bars1 = ax.bar(
    x - width, correlations["FoGro"], width, label="FoGro", color="blue", alpha=0.7
)
bars2 = ax.bar(x, correlations["Random"], width, label="Random", color="red", alpha=0.7)
bars3 = ax.bar(
    x + width, correlations["Full"], width, label="Full", color="green", alpha=0.7
)

ax.set_xlabel("Layer")
ax.set_ylabel("Spearman Correlation (ρ)")
ax.set_title(
    "Correlation: Running Variance vs Channel Index\n(Positive = later channels have higher variance)"
)
ax.set_xticks(x)
ax.set_xticklabels(layer_names, rotation=45, ha="right")
ax.legend()
ax.axhline(y=0, color="black", linestyle="-", linewidth=0.5)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print(f"\nMean correlation across layers:")
print(f"  FoGro:  {np.mean(correlations['FoGro']):+.3f}")
print(f"  Random: {np.mean(correlations['Random']):+.3f}")
print(f"  Full:   {np.mean(correlations['Full']):+.3f}")

## Key Findings

### Running Variance Evolution Across Channels

The analysis reveals a **striking pattern** in how `running_var` evolves from early channels to later channels:

1. **FoGro Model** (mean ρ = -0.87):
   - Shows the **strongest negative correlation** between channel index and running variance
   - Earlier channels (added first during growth) have significantly **higher variance**
   - Later channels (added more recently) have **lower, more uniform variance** (~1.0)
   - This suggests earlier channels have been exposed to more data and developed more distinctive features

2. **Random Model** (mean ρ = -0.65):
   - Also shows a negative correlation, but **weaker than FoGro**
   - Similar pattern: earlier channels have higher variance
   - The effect is less pronounced, possibly because random initialization doesn't optimize channel ordering

3. **Full Model** (mean ρ ≈ 0):
   - Shows **no systematic correlation** between channel index and variance
   - Running variance is **relatively uniform** across all channels (~1.0-2.0)
   - This is expected for a standard trained model where all channels are treated equally

### Interpretation

The **channel order matters** in growing networks:
- In FoGro/Random models, channels are added incrementally during training
- **Earlier channels** have been trained longer and develop more varied/specialized activations
- **Later channels** are newer and haven't accumulated as much variance in their statistics
- This creates a gradient from "mature" (high variance) to "immature" (low variance) channels

## Other BatchNorm Attributes

Let's analyze the same pattern for `running_mean`, `weight` (γ), and `bias` (β).

In [ ]:
def plot_bn_attribute(attribute_name, ylabel, title_suffix=""):
    """Plot a BatchNorm attribute across channels for all models."""
    fig, axes = plt.subplots(4, 2, figsize=(14, 16))
    axes = axes.flatten()

    layer_names = list(fogro_stats.keys())

    for idx, layer_name in enumerate(layer_names):
        ax = axes[idx]

        fogro_val = fogro_stats[layer_name][attribute_name]
        random_val = random_stats[layer_name][attribute_name]
        full_val = full_stats[layer_name][attribute_name]

        fogro_channels = np.arange(len(fogro_val))
        random_channels = np.arange(len(random_val))
        full_channels = np.arange(len(full_val))

        ax.plot(
            fogro_channels,
            fogro_val,
            "b-",
            label=f"FoGro ({len(fogro_val)} ch)",
            alpha=0.8,
            linewidth=1.5,
        )
        ax.plot(
            random_channels,
            random_val,
            "r-",
            label=f"Random ({len(random_val)} ch)",
            alpha=0.8,
            linewidth=1.5,
        )
        ax.plot(
            full_channels,
            full_val,
            "g-",
            label=f"Full ({len(full_val)} ch)",
            alpha=0.8,
            linewidth=1.5,
        )

        ax.set_xlabel("Channel Index")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{layer_name}")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.suptitle(f"{ylabel} Evolution Across Channels{title_suffix}", y=1.02, fontsize=14)
    plt.show()

    return fig

### Running Mean

In [ ]:
plot_bn_attribute("running_mean", "Running Mean")

### Weight (γ) - Scale Parameter

In [ ]:
plot_bn_attribute("weight", "Weight (γ)")

### Bias (β) - Shift Parameter

In [ ]:
plot_bn_attribute("bias", "Bias (β)")

### Correlation Summary for All Attributes

In [ ]:
# Correlation analysis for all BatchNorm attributes
from scipy import stats

attributes = ["running_var", "running_mean", "weight", "bias"]
all_correlations = {attr: {"FoGro": [], "Random": [], "Full": []} for attr in attributes}

for layer_name in fogro_stats.keys():
    for attr in attributes:
        fogro_val = fogro_stats[layer_name][attr]
        random_val = random_stats[layer_name][attr]
        full_val = full_stats[layer_name][attr]

        fogro_channels = np.arange(len(fogro_val))
        random_channels = np.arange(len(random_val))
        full_channels = np.arange(len(full_val))

        fogro_corr, _ = stats.spearmanr(fogro_channels, fogro_val)
        random_corr, _ = stats.spearmanr(random_channels, random_val)
        full_corr, _ = stats.spearmanr(full_channels, full_val)

        all_correlations[attr]["FoGro"].append(fogro_corr)
        all_correlations[attr]["Random"].append(random_corr)
        all_correlations[attr]["Full"].append(full_corr)

# Create summary bar plot
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(attributes))
width = 0.25

mean_corrs = {
    model: [np.mean(all_correlations[attr][model]) for attr in attributes]
    for model in ["FoGro", "Random", "Full"]
}

bars1 = ax.bar(
    x - width, mean_corrs["FoGro"], width, label="FoGro", color="blue", alpha=0.7
)
bars2 = ax.bar(x, mean_corrs["Random"], width, label="Random", color="red", alpha=0.7)
bars3 = ax.bar(
    x + width, mean_corrs["Full"], width, label="Full", color="green", alpha=0.7
)

ax.set_xlabel("BatchNorm Attribute")
ax.set_ylabel("Mean Spearman Correlation (ρ)")
ax.set_title(
    "Mean Correlation: Attribute Value vs Channel Index\n(Negative = earlier channels have higher/different values)"
)
ax.set_xticks(x)
ax.set_xticklabels(["Running Var", "Running Mean", "Weight (γ)", "Bias (β)"])
ax.legend()
ax.axhline(y=0, color="black", linestyle="-", linewidth=0.5)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

# Print summary table
print("\nMean Correlation (ρ) across all layers:")
print("-" * 60)
print(f"{'Attribute':<15} {'FoGro':>10} {'Random':>10} {'Full':>10}")
print("-" * 60)
for attr in attributes:
    fogro_mean = np.mean(all_correlations[attr]["FoGro"])
    random_mean = np.mean(all_correlations[attr]["Random"])
    full_mean = np.mean(all_correlations[attr]["Full"])
    print(f"{attr:<15} {fogro_mean:>+10.3f} {random_mean:>+10.3f} {full_mean:>+10.3f}")

## Summary of All BatchNorm Attributes

| Attribute | FoGro (ρ) | Random (ρ) | Full (ρ) | Pattern in Growing Networks |
|-----------|-----------|------------|----------|------------------------------|
| **running_var** | -0.87 | -0.65 | ~0 | Earlier channels have **higher variance** |
| **running_mean** | +0.33 | +0.35 | ~0 | Earlier channels have **lower/more negative mean** |
| **weight (γ)** | -0.44 | -0.33 | ~0 | Earlier channels have **higher scale factors** |
| **bias (β)** | +0.46 | +0.42 | ~0 | Earlier channels have **more negative bias** |

### Interpretation

The growing network pattern is **consistent across all BatchNorm parameters**:

1. **Earlier channels** (added first):
   - Higher running variance → more varied activations
   - More negative running mean → shifted activation distributions
   - Higher weight (γ) → learned to amplify their activations
   - More negative bias (β) → learned specific offset adjustments

2. **Later channels** (added recently):
   - Lower variance (~1.0) → closer to initialization
   - Running mean closer to 0 → less shifted distributions
   - Weight (γ) closer to 1 → near-identity scaling
   - Bias (β) closer to 0 → minimal offset

3. **Full model** shows **no such pattern** (ρ ≈ 0 for all attributes), confirming this is a growth-specific phenomenon.